# Лабораторная работа №1. Землетрясения USGS

**Марюшкина Софья, Чуева Арина — поток 1.3.**

Источник: [USGS Earthquake Catalog API](https://earthquake.usgs.gov/fdsnws/event/1/1). Работа включает сбор событий с веб-ресурса, сохранение и догрузку в SQLite, а также шесть агрегаций двумя способами: pandas и SQLAlchemy.


## 1. Получение данных

Выборка: землетрясения магнитудой от 2,5 за 1–14 января 2025 года (UTC). Два недельных интервала позволяют отдельно проверить догрузку. USGS возвращает GeoJSON: каждое событие содержит свойства и координаты.


In [ ]:
import json
from datetime import date, datetime, time as dt_time, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from sqlalchemy import Column, Float, Integer, MetaData, String, Table, case, create_engine, func, select
from sqlalchemy.dialects.sqlite import insert as sqlite_insert
from IPython.display import display

API_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
MIN_MAGNITUDE = 2.5
PAGE_SIZE = 20_000
DB_PATH = Path("data/earthquakes.sqlite")
RESULTS_DIR = Path("results")


In [ ]:
def get_events(start: date, end: date) -> list[dict]:
    """Получить события за период [start, end) с учётом страниц API."""
    end_inclusive = datetime.combine(end, dt_time.min, tzinfo=timezone.utc) - timedelta(milliseconds=1)
    offset = 1
    events = []

    while True:
        params = {
            "format": "geojson",
            "eventtype": "earthquake",
            "starttime": start.isoformat(),
            "endtime": end_inclusive.isoformat(timespec="milliseconds"),
            "minmagnitude": MIN_MAGNITUDE,
            "orderby": "time-asc",
            "limit": PAGE_SIZE,
            "offset": offset,
        }
        request = Request(
            f"{API_URL}?{urlencode(params)}",
            headers={"User-Agent": "pda-lab1-usgs-earthquakes/1.0 (educational project)"},
        )
        with urlopen(request, timeout=60) as response:
            if response.status == 204:
                break
            page = json.load(response)["features"]

        events.extend(page)
        if len(page) < PAGE_SIZE:
            break
        offset += len(page)

    return events

first_features = get_events(date(2025, 1, 1), date(2025, 1, 8))
print("Получено за 1–7 января:", len(first_features))
display(pd.json_normalize(first_features[:3])[["id", "properties.mag", "properties.place"]])


## 2. Преобразование и сохранение в SQLite

Из GeoJSON извлекаются поля события. В массиве координат USGS порядок значений — долгота, широта, глубина. Время переводится из миллисекунд Unix в UTC.


In [ ]:
def to_rows(features: list[dict]) -> list[dict]:
    rows = []
    for feature in features:
        props = feature["properties"]
        coordinates = feature["geometry"]["coordinates"]
        if feature.get("id") is None or props.get("mag") is None or len(coordinates) < 2:
            continue

        rows.append({
            "event_id": str(feature["id"]),
            "occurred_at": datetime.fromtimestamp(
                props["time"] / 1000, tz=timezone.utc
            ).isoformat(timespec="milliseconds"),
            "updated_ms": int(props["updated"]),
            "magnitude": float(props["mag"]),
            "depth_km": coordinates[2] if len(coordinates) > 2 else None,
            "latitude": coordinates[1],
            "longitude": coordinates[0],
            "place": props.get("place"),
            "network": props.get("net"),
            "status": props.get("status"),
            "tsunami": int(props.get("tsunami") or 0),
            "significance": props.get("sig"),
            "url": props.get("url"),
        })
    return rows

first_rows = to_rows(first_features)
display(pd.DataFrame(first_rows).head())


In [ ]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
engine = create_engine(f"sqlite:///{DB_PATH.resolve().as_posix()}")
metadata = MetaData()
earthquakes = Table(
    "earthquakes", metadata,
    Column("event_id", String, primary_key=True),
    Column("occurred_at", String, nullable=False),
    Column("updated_ms", Integer, nullable=False),
    Column("magnitude", Float, nullable=False),
    Column("depth_km", Float),
    Column("latitude", Float),
    Column("longitude", Float),
    Column("place", String),
    Column("network", String),
    Column("status", String),
    Column("tsunami", Integer),
    Column("significance", Integer),
    Column("url", String),
)
metadata.create_all(engine)

with engine.connect() as connection:
    print("Строк в SQLite до загрузки:", connection.scalar(select(func.count()).select_from(earthquakes)))


Повторная запись того же набора не должна создавать дубли. Первичный ключ `event_id` определяет событие; upsert обновляет его только при более позднем `updated_ms`.


In [ ]:
def load_rows(rows: list[dict]) -> int:
    if not rows:
        return 0

    insert_stmt = sqlite_insert(earthquakes)
    update_columns = {
        column.name: getattr(insert_stmt.excluded, column.name)
        for column in earthquakes.columns
        if column.name != "event_id"
    }
    upsert_stmt = insert_stmt.on_conflict_do_update(
        index_elements=[earthquakes.c.event_id],
        set_=update_columns,
        where=insert_stmt.excluded.updated_ms > earthquakes.c.updated_ms,
    )

    written = 0
    with engine.begin() as connection:
        for start in range(0, len(rows), 1000):
            written += connection.execute(upsert_stmt, rows[start:start + 1000]).rowcount
    return written

print("Записано при первой загрузке:", load_rows(first_rows))
print("Записано при повторе того же набора:", load_rows(first_rows))
with engine.connect() as connection:
    print("Строк в SQLite:", connection.scalar(select(func.count()).select_from(earthquakes)))


In [ ]:
second_features = get_events(date(2025, 1, 8), date(2025, 1, 15))
second_rows = to_rows(second_features)
print("Получено за 8–14 января:", len(second_rows))
print("Добавлено или обновлено:", load_rows(second_rows))
with engine.connect() as connection:
    print("Всего строк в SQLite:", connection.scalar(select(func.count()).select_from(earthquakes)))


## 3. Шесть агрегаций в pandas

pandas считывает накопленные строки из SQLite в DataFrame и группирует их в памяти.


In [ ]:
with engine.connect() as connection:
    events = pd.read_sql_query(select(earthquakes), connection)

events["day_utc"] = events["occurred_at"].str[:10]
events["hour_utc"] = events["occurred_at"].str[11:13]
events["magnitude_band"] = pd.cut(
    events["magnitude"], [-float("inf"), 3, 4, 5, 6, float("inf")],
    labels=["M < 3", "3 ≤ M < 4", "4 ≤ M < 5", "5 ≤ M < 6", "M ≥ 6"],
    right=False,
)
events["depth_band"] = pd.cut(
    events["depth_km"], [-float("inf"), 70, 300, float("inf")],
    labels=["мелкие (<70 км)", "промежуточные (70–300 км)", "глубокие (≥300 км)"],
    right=False,
)
print("Строк для анализа:", len(events))
display(events.head())


### 3.1. Число событий и магнитуда по дням


In [ ]:
pd_daily = (
    events.groupby("day_utc", observed=True)
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"),
         max_magnitude=("magnitude", "max"))
    .reset_index()
    .sort_values("day_utc")
)
display(pd_daily)


### 3.2. Распределение по диапазонам магнитуды


In [ ]:
pd_magnitude = (
    events.groupby("magnitude_band", observed=True)
    .agg(events=("event_id", "size"), avg_depth_km=("depth_km", "mean"))
    .reset_index()
)
display(pd_magnitude)


### 3.3. Распределение по глубине


In [ ]:
pd_depth = (
    events.dropna(subset=["depth_km"])
    .groupby("depth_band", observed=True)
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"))
    .reset_index()
)
display(pd_depth)


### 3.4. Статистика по сетям каталога


In [ ]:
pd_network = (
    events.groupby("network", dropna=False)
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"),
         max_magnitude=("magnitude", "max"))
    .reset_index()
    .sort_values(["events", "network"], ascending=[False, True])
)
display(pd_network)


### 3.5. Распределение по часам UTC


In [ ]:
pd_hour = (
    events.groupby("hour_utc")
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"))
    .reset_index()
    .sort_values("hour_utc")
)
display(pd_hour)


### 3.6. Сравнение по флагу tsunami


In [ ]:
pd_tsunami = (
    events.groupby("tsunami")
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"),
         avg_significance=("significance", "mean"))
    .reset_index()
    .sort_values("tsunami")
)
display(pd_tsunami)


## 4. Те же шесть агрегаций в SQLAlchemy

SQLAlchemy формирует запросы, которые выполняются внутри SQLite. Результаты возвращаются в виде DataFrame для сопоставления с pandas.


In [ ]:
e = earthquakes.c
day_sql = func.substr(e.occurred_at, 1, 10).label("day_utc")
hour_sql = func.substr(e.occurred_at, 12, 2).label("hour_utc")
magnitude_band_sql = case(
    (e.magnitude < 3, "M < 3"),
    (e.magnitude < 4, "3 ≤ M < 4"),
    (e.magnitude < 5, "4 ≤ M < 5"),
    (e.magnitude < 6, "5 ≤ M < 6"),
    else_="M ≥ 6",
).label("magnitude_band")
depth_band_sql = case(
    (e.depth_km < 70, "мелкие (<70 км)"),
    (e.depth_km < 300, "промежуточные (70–300 км)"),
    else_="глубокие (≥300 км)",
).label("depth_band")


### 4.1. Число событий и магнитуда по дням


In [ ]:
query = (
    select(day_sql, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"),
           func.max(e.magnitude).label("max_magnitude"))
    .group_by(day_sql)
    .order_by(day_sql)
)
with engine.connect() as connection:
    sql_daily = pd.read_sql_query(query, connection)
display(sql_daily)


### 4.2. Распределение по диапазонам магнитуды


In [ ]:
query = (
    select(magnitude_band_sql, func.count().label("events"),
           func.avg(e.depth_km).label("avg_depth_km"))
    .group_by(magnitude_band_sql)
    .order_by(magnitude_band_sql)
)
with engine.connect() as connection:
    sql_magnitude = pd.read_sql_query(query, connection)
display(sql_magnitude)


### 4.3. Распределение по глубине


In [ ]:
query = (
    select(depth_band_sql, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"))
    .where(e.depth_km.is_not(None))
    .group_by(depth_band_sql)
    .order_by(depth_band_sql)
)
with engine.connect() as connection:
    sql_depth = pd.read_sql_query(query, connection)
display(sql_depth)


### 4.4. Статистика по сетям каталога


In [ ]:
query = (
    select(e.network, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"),
           func.max(e.magnitude).label("max_magnitude"))
    .group_by(e.network)
    .order_by(func.count().desc(), e.network)
)
with engine.connect() as connection:
    sql_network = pd.read_sql_query(query, connection)
display(sql_network)


### 4.5. Распределение по часам UTC


In [ ]:
query = (
    select(hour_sql, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"))
    .group_by(hour_sql)
    .order_by(hour_sql)
)
with engine.connect() as connection:
    sql_hour = pd.read_sql_query(query, connection)
display(sql_hour)


### 4.6. Сравнение по флагу tsunami


In [ ]:
query = (
    select(e.tsunami, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"),
           func.avg(e.significance).label("avg_significance"))
    .group_by(e.tsunami)
    .order_by(e.tsunami)
)
with engine.connect() as connection:
    sql_tsunami = pd.read_sql_query(query, connection)
display(sql_tsunami)


## 5. Сопоставление результатов

Для каждой агрегации сравниваются группы, количества и числовые показатели. Производные таблицы сохраняются в CSV.


In [ ]:
pairs = {
    "01_daily": (pd_daily, sql_daily),
    "02_magnitude": (pd_magnitude, sql_magnitude),
    "03_depth": (pd_depth, sql_depth),
    "04_network": (pd_network, sql_network),
    "05_hour": (pd_hour, sql_hour),
    "06_tsunami": (pd_tsunami, sql_tsunami),
}

def comparable(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    key = result.columns[0]
    result[key] = result[key].astype(str)
    return result.sort_values(key).reset_index(drop=True)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name, (pandas_result, sql_result) in pairs.items():
    pd.testing.assert_frame_equal(
        comparable(pandas_result), comparable(sql_result),
        check_dtype=False, atol=1e-8, rtol=1e-8,
    )
    pandas_result.to_csv(RESULTS_DIR / f"{name}_pandas.csv", index=False, encoding="utf-8-sig")
    sql_result.to_csv(RESULTS_DIR / f"{name}_sql.csv", index=False, encoding="utf-8-sig")
    print(name, "— результаты совпадают")

print("SQLite:", DB_PATH.resolve())
print("CSV:", RESULTS_DIR.resolve())


## Вывод

Данные USGS собраны напрямую через API и сохранены в SQLite. Первичный ключ и upsert позволяют повторять загрузку без создания дублей и принимать уточнения каталога. Шесть разных аналитических вопросов реализованы в pandas и SQLAlchemy; полученные результаты сопоставлены. Численные итоги относятся к указанной выборке и могут измениться после обновлений каталога USGS.
